# 02 - LLaMEA Single-Problem Evolution & Analysis

This notebook focuses on **inspecting system prompts**, **configuring LLM providers**, **running LLaMEA evolutionary search for a single target problem** (`run_evolution_for_problem`), **saving experiment summaries**, and **visualizing convergence curves** to verify everything works properly before running larger batch experiments.

In [ ]:
# Setup paths and imports
import sys
from pathlib import Path
# Add project root src/ to sys.path regardless of directory name or launch folder
root_dir = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
src_path = str(root_dir / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import json
import os
import plotly.graph_objects as go
from dotenv import load_dotenv

from problems.bbob import BBOBProblem
from llm.prompts import TASK_PROMPT_NOISY, TASK_PROMPT_CLEAN, EXAMPLE_PROMPT, FORMAT_PROMPT
from llm.providers import build_llm, Provider
from core.runner import run_evolution_for_problem
from analysis.results import save_summary, print_experiment_summary


## 1. Unified Single-Problem Configuration
Define configuration variables for the target problem ID, dimensionality, noise standard deviation, evaluation budget, and evolution iterations.

In [ ]:
problem_id = 1
dim = 3
instance_id = 1
noise_std = 0.05
iterations = 5
budget = 1000
mode = "noisy"

problem = BBOBProblem(problem_id=problem_id, dim=dim, instance_id=instance_id)

print(f"Loaded Target Problem: BBOB {problem_id} ({dim}D, Instance {instance_id})")
print(f"Domain Bounds: [{problem.lower_bound}, {problem.upper_bound}]")
print(f"True Optimum: {problem.true_optimum:.4f}")
print(f"Experiment Configuration: mode={mode}, noise_std={noise_std}, iterations={iterations}, budget={budget}")


## 2. Inspect System Prompts
Review the prompt templates passed to the LLM during LLaMEA evolution for the configured problem.

In [ ]:
print("=== NOISY TASK PROMPT (BBOB f1, 3D) ===")
print(TASK_PROMPT_NOISY.format(
    problem_id=problem.problem_id,
    dim=problem.dim,
    lower_bound=problem.lower_bound,
    upper_bound=problem.upper_bound
))
print("\n=== CLEAN TASK PROMPT (BBOB f1, 3D) ===")
print(TASK_PROMPT_CLEAN.format(
    problem_id=problem.problem_id,
    dim=problem.dim,
    lower_bound=problem.lower_bound,
    upper_bound=problem.upper_bound
))
print("\n=== FORMAT PROMPT ===")
print(FORMAT_PROMPT)


## 3. LLM Provider Configuration
Configure LLM provider (e.g. Gemini or LM Studio) from environment variables or `.env` file.

In [ ]:
env_path = root_dir / ".env"
if env_path.exists():
    load_dotenv(dotenv_path=env_path)

llm_provider = os.getenv("LLM_PROVIDER")

try:
    llm = build_llm(llm_provider)
    print("=================================================================")
    print(f"Initialized Provider: {llm_provider}")
    print(f"Model Target:         {getattr(llm, 'model', 'unknown')}")
    print("=================================================================")
except Exception as e:
    print(f"Provider setup notice: {e}")
    llm = None


## 4. Execute LLaMEA Evolutionary Search & Save Summary
Run LLaMEA for the configured single problem, then explicitly save the summary JSON artifact.

In [ ]:
optimizer = run_evolution_for_problem(
    problem=problem,
    llm=llm,
    budget=budget,
    iterations=iterations,
    mode=mode,
    noise_std=noise_std,
    log=True
)

# Save summary artifact so it can be analyzed
experiment_name = f"bbob_{problem.problem_id}_dim{problem.dim}_{mode}"
summary_file = save_summary(
    history=optimizer.run_history,
    problem_id=problem.problem_id,
    dim=problem.dim,
    output_dir=root_dir / "logs" / experiment_name,
    mode=mode
)
print(f"Evolution completed! Saved summary to: {summary_file}")


## 5. Load & Analyze Experiment Results
Collect saved summaries from `logs/` directory using `print_experiment_summary`, load `summary.json`, and plot the convergence curve.

In [ ]:
logs_dir = root_dir / "logs"
print_experiment_summary(logs_dir)

all_experiments = [p.name for p in logs_dir.glob("*") if p.is_dir()]

if all_experiments:
    target_exp = f"bbob_{problem.problem_id}_dim{problem.dim}_{mode}"
    if not (logs_dir / target_exp).exists() and all_experiments:
        target_exp = all_experiments[0]
        
    summary_file = logs_dir / target_exp / "summary.json"
    if summary_file.exists():
        with open(summary_file) as f:
            data = json.load(f)
        print(f"\nAnalyzing Experiment: {target_exp}")
        print(f"Best Candidate Algorithm: {data.get('best_algorithm')}")
        print(f"Best Final Error: {data.get('best_final_error')}")
        
        # Convergence Plotly Chart
        history = data.get("iterations", [])
        iterations_x = [h["iteration"] for h in history]
        errors_y = [h.get("final_error", float('nan')) for h in history]
        
        fig = go.Figure()
        fig.add_trace(go.Scatter(
            x=iterations_x,
            y=errors_y,
            mode='lines+markers',
            name='Final Error',
            line=dict(color='#10B981', width=3)
        ))
        fig.update_layout(
            title=f"LLaMEA Evolutionary Convergence ({target_exp})",
            xaxis_title="Iteration",
            yaxis_title="Final Error from Optimum",
            template="plotly_white",
            height=450,
            width=850
        )
        fig.show()
